In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras import backend as K
import random

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = x_train.reshape((-1, 784))
x_test = x_test.reshape((-1, 784))

In [ ]:
def create_pairs(x, digit_indices):
    """
    Creates positive and negative pairs.
    Returns: pairs (numpy array), labels (numpy array)
    """
    pairs = []
    labels = []

    n = min([len(digit_indices[d]) for d in range(10)]) - 1

    for d in range(10):
        for i in range(n):
            # 1. Positive Pair (Same Class)
            z1, z2 = digit_indices[d][i], digit_indices[d][i + 1]
            pairs += [[x[z1], x[z2]]]

            # 2. Negative Pair (Different Class)
            # Pick a random digit that is NOT d
            inc = random.randrange(1, 10)
            dn = (d + inc) % 10
            z1, z2 = digit_indices[d][i], digit_indices[dn][i]
            pairs += [[x[z1], x[z2]]]

            labels += [0, 1] # 0 for Similar, 1 for Dissimilar

    return np.array(pairs), np.array(labels)

In [ ]:
digit_indices = [np.where(y_train == i)[0] for i in range(10)]
tr_pairs, tr_y = create_pairs(x_train, digit_indices)

digit_indices_test = [np.where(y_test == i)[0] for i in range(10)]
te_pairs, te_y = create_pairs(x_test, digit_indices_test)

print(f"Training pairs shape: {tr_pairs.shape}")

Training pairs shape: (108400, 2, 784)


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, Lambda

def euclidean_distance(vects):
    x, y = vects
    sum_square = K.sum(K.square(x - y), axis=1, keepdims=True)
    # Add epsilon to prevent NaN gradients (div by zero)
    return K.sqrt(K.maximum(sum_square, K.epsilon()))

def create_base_network(input_shape):
    """
    The Fully Connected 'Leg' of the Siamese Network
    """
    input = Input(shape=input_shape)
    x = Dense(128, activation='relu')(input)
    x = Dropout(0.1)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.1)(x)
    # The Embedding Layer: This maps the image to a vector of size 128
    x = Dense(128, activation=None)(x)
    return Model(input, x)

# 1. Define Shapes
input_shape = (784,)

# 2. Create the Base Network (Shared Weights)
base_network = create_base_network(input_shape)

# 3. Define the Two Inputs
input_a = Input(shape=input_shape)
input_b = Input(shape=input_shape)

# 4. Connect Inputs to Base Network
# Note: We call base_network twice, but it's the SAME object.
# This enforces weight sharing.
processed_a = base_network(input_a)
processed_b = base_network(input_b)

# 5. Calculate Distance
distance = Lambda(euclidean_distance)([processed_a, processed_b])

# 6. Final Model
model = Model([input_a, input_b], distance)

In [ ]:
def contrastive_loss(y_true, y_pred):
    """
    y_true: 0 = similar, 1 = dissimilar
    y_pred: euclidean distance predicted by model
    """
    margin = 1.0
    square_pred = K.square(y_pred)
    margin_square = K.square(K.maximum(margin - y_pred, 0))

    # Formula: (1 - Y) * D^2 + (Y) * max(0, m - D)^2
    # Note: K.mean averages the loss over the batch
    return K.mean((1 - y_true) * square_pred + (y_true) * margin_square)

In [ ]:
# Compile
model.compile(loss=contrastive_loss, optimizer='adam')

# Train
history = model.fit(
    [tr_pairs[:, 0], tr_pairs[:, 1]], tr_y,
    batch_size=128,
    epochs=10, # Increase epochs for better accuracy
    validation_data=([te_pairs[:, 0], te_pairs[:, 1]], te_y)
)

Epoch 1/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 14s 14ms/step - loss: 0.2339 - val_loss: 0.0749
Epoch 2/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 0.0877 - val_loss: 0.0561
Epoch 3/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.0694 - val_loss: 0.0485
Epoch 4/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.0599 - val_loss: 0.0427
Epoch 5/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - loss: 0.0542 - val_loss: 0.0399
Epoch 6/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 0.0499 - val_loss: 0.0385
Epoch 7/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 0.0470 - val_loss: 0.0353
Epoch 8/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.0442 - val_loss: 0.0351
Epoch 9/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - loss: 0.0420 - val_loss: 0.0336
Epoch 10/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - loss: 0.0399 - val_loss: 0.0339


In [ ]:
import os
from tensorflow.keras.models import load_model

# Define the filename
file_name = "my_siamese_model.h5"

# CHECK: Does the saved model exist?
if os.path.exists(file_name):
    print(f"Found saved model '{file_name}'. Loading it now...")
    print("Skipping training to save time.")

    # LOAD the model with the custom math functions
    model = load_model(
        file_name,
        custom_objects={
            'contrastive_loss': contrastive_loss,
            'euclidean_distance': euclidean_distance
        }
    )

else:
    print(f"No saved model found. Starting training...")

    # TRAIN the model (Only runs if file is missing)
    history = model.fit(
        [tr_pairs[:, 0], tr_pairs[:, 1]], tr_y,
        batch_size=128,
        epochs=10,
        validation_data=([te_pairs[:, 0], te_pairs[:, 1]], te_y)
    )

    # SAVE it so we don't have to train next time
    model.save(file_name)
    print(f"Training finished. Model saved as {file_name}")

# ==========================================
# CONTINUE TO STEP 5 (TESTING)
# ==========================================
print("Proceeding to evaluation...")
# Your testing code (test_one_shot, etc.) goes here...

No saved model found. Starting training...
Epoch 1/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 0.0368 - val_loss: 0.0337
Epoch 2/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0351 - val_loss: 0.0330
Epoch 3/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 0.0338 - val_loss: 0.0328
Epoch 4/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 0.0335 - val_loss: 0.0321
Epoch 5/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0327 - val_loss: 0.0334
Epoch 6/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 0.0319 - val_loss: 0.0324
Epoch 7/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 0.0310 - val_loss: 0.0324
Epoch 8/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - loss: 0.0305 - val_loss: 0.0321
Epoch 9/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0304 - val_loss: 0.0318
Epoch 10/10
847/847 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - loss: 0.0297 - val_loss: 0.0316


Training finished. Model saved as my_siamese_model.h5
Proceeding to evaluation...


In [ ]:
def compute_accuracy(y_true, y_pred):
    """Compute classification accuracy with a fixed threshold on distances."""
    pred = y_pred.ravel() < 0.5
    return np.mean(pred == y_true)

# Function to test One-Shot Learning performance
def test_one_shot(model, N, k, verbose=True):
    """
    N: Number of ways (classes) - for MNIST usually 10
    k: Number of trials
    """
    n_correct = 0

    if verbose:
        print(f"Evaluating {k} random {N}-way one-shot learning tasks...")

    for i in range(k):
        # 1. Select a random character from test set to be the 'Query'
        # We assume digit_indices_test is available from Step 1
        true_class = random.randint(0, N-1)
        # Get index of a random image of this class
        idx_query = random.choice(digit_indices_test[true_class])
        query_image = x_test[idx_query]

        # 2. Build Support Set (1 image per class)
        support_set = []
        for class_idx in range(N):
            # If it's the true class, pick a DIFFERENT image than the query
            if class_idx == true_class:
                # simple logic to pick a different index
                idx_support = random.choice(digit_indices_test[class_idx])
                # Ensure we didn't pick the exact same image (unlikely but possible)
            else:
                idx_support = random.choice(digit_indices_test[class_idx])

            support_set.append(x_test[idx_support])

        support_set = np.array(support_set)

        # 3. Create Pairs for Prediction
        # Replicate query image N times
        query_replicated = np.array([query_image] * N)

        # 4. Predict Distances
        distances = model.predict([query_replicated, support_set], verbose=0)

        # 5. Check if the smallest distance corresponds to the true class
        if np.argmin(distances) == true_class:
            n_correct += 1

    percent_correct = (100.0 * n_correct / k)
    if verbose:
        print(f"Got {n_correct}/{k} correct. {percent_correct}% accuracy")
    return percent_correct

# Run the evaluation
accuracy = test_one_shot(model, N=10, k=200)

Evaluating 200 random 10-way one-shot learning tasks...
Got 188/200 correct. 94.0% accuracy


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def classify_with_details(model, query_img, support_set_images, support_set_labels):
    """
    Classifies a query image and prints the similarity score (distance)
    for every image in the support set.
    """

    # 1. Prepare Inputs
    # We need to repeat the query image to match the number of support images
    n_support = len(support_set_images)
    query_duplicated = np.array([query_img] * n_support)

    # 2. Calculate Distances (The "Similarity Scores")
    # This runs the Siamese network on (Query vs Support_1), (Query vs Support_2)...
    distances = model.predict([query_duplicated, support_set_images], verbose=0)

    # Flatten the distances for easier reading
    distances = distances.flatten()

    # 3. Print Detailed Report
    print(f"{'-'*40}")
    print(f"| {'Class Label':<12} | {'Distance':<22} |")
    print(f"{'-'*40}")

    # Store results to find the best one
    results = []

    for i in range(n_support):
        label = support_set_labels[i]
        dist = distances[i]
        results.append((label, dist))

        # Visual indicator for the score
        print(f"| {str(label):<12} | {dist:.4f}")

    print(f"{'-'*40}")

    # 4. Find the Highest Similarity (Lowest Distance)
    # The winner is the one with the MINIMUM distance
    best_match_idx = np.argmin(distances)
    predicted_label = support_set_labels[best_match_idx]
    confidence_score = distances[best_match_idx]

    print(f"\n✅ RESULT: The query image is classified as: {predicted_label}")
    print(f"   (Closest match with distance: {confidence_score:.4f})\n")

    return predicted_label

# --- USAGE EXAMPLE ---

# 1. Pick a random query image from the Test Set
import random
query_idx = random.randint(0, len(x_test)-1)
query_img = x_test[query_idx]
actual_label = y_test[query_idx]

print(f"🔎 Query Image True Label: {actual_label}")

# 2. Build a Support Set (One image for each digit 0-9)
# We assume 'digit_indices_test' exists from the previous steps
support_imgs = []
support_labels = []

for i in range(10):
    # Pick one random image for this digit
    idx = random.choice(digit_indices_test[i])
    support_imgs.append(x_test[idx])
    support_labels.append(i)

support_imgs = np.array(support_imgs)

# 3. Run the detailed classification
prediction = classify_with_details(model, query_img, support_imgs, support_labels)

# 4. (Optional) Verification
if prediction == actual_label:
    print("SUCCESS: The model predicted correctly!")
else:
    print("FAILURE: The model predicted incorrectly.")

🔎 Query Image True Label: 0
----------------------------------------
| Class Label  | Distance               |
----------------------------------------
| 0            | 0.0178
| 1            | 1.4293
| 2            | 0.9766
| 3            | 1.1045
| 4            | 0.9565
| 5            | 0.8577
| 6            | 0.8907
| 7            | 1.0367
| 8            | 0.8829
| 9            | 0.9991
----------------------------------------

✅ RESULT: The query image is classified as: 0
   (Closest match with distance: 0.0178)

SUCCESS: The model predicted correctly!
